# forced_alignment

> Forced alignment service for audio-informed text pre-splitting via forced alignment plugin

In [ ]:
#| default_exp services.forced_alignment

In [ ]:
#| export
import re
import asyncio
from typing import List, Dict, Any, Optional, Tuple

from cjm_plugin_system.core.manager import PluginManager

from cjm_transcription_plugin_system.forced_alignment_core import ForcedAlignItem, ForcedAlignResult
from cjm_transcript_segmentation.models import TextSegment
from cjm_transcript_vad_align.models import VADChunk

## Word-to-Text Mapping

The forced alignment model strips punctuation from words. We need to map each
stripped word back to its position in the original text to preserve punctuation.

**Example:**
```
Original text:  "November the 10th, Wednesday, 9 p.m."
FA words:       ["November", "the", "10th", "Wednesday", "9", "pm"]
Mapped:         [(0, 8), (9, 12), (13, 18), (19, 29), (30, 31), (32, 36)]
                → char spans into original text
```

The approach: for each FA word, find where it matches in the remaining original
text (stripping punctuation from the original for comparison), then record the
full span including any trailing punctuation.

In [ ]:
#| export
# Regex to strip punctuation for comparison (matches what FA models strip)
_PUNCT_RE = re.compile(r'[^\w\s]', re.UNICODE)


def _strip_punct(text: str) -> str:
    """Strip punctuation from text for comparison with FA output."""
    return _PUNCT_RE.sub('', text)

In [ ]:
#| export
def map_fa_words_to_text(
    text: str,  # Original text with punctuation
    fa_items: List[ForcedAlignItem],  # FA word-level alignment results
) -> List[Tuple[int, int]]:  # List of (start_char, end_char) spans into original text
    """Map forced alignment words back to character spans in the original text.
    
    Walks through the original text, matching each FA word (punctuation-stripped)
    against original text tokens. Returns character offset pairs for each FA word.
    """
    spans = []
    pos = 0  # Current position in original text
    
    for item in fa_items:
        fa_word = item.text.lower()
        
        # Skip whitespace to find next token start
        while pos < len(text) and text[pos].isspace():
            pos += 1
        
        if pos >= len(text):
            break
        
        # Find the end of the current token in original text
        # A token is a run of non-space characters
        token_start = pos
        token_end = pos
        while token_end < len(text) and not text[token_end].isspace():
            token_end += 1
        
        orig_token = text[token_start:token_end]
        stripped_token = _strip_punct(orig_token).lower()
        
        # Check if FA word matches this token (after stripping punctuation)
        if stripped_token == fa_word:
            spans.append((token_start, token_end))
            pos = token_end
        else:
            # Handle multi-token FA words (e.g., "p.m." → "pm") or
            # cases where the original has punctuation splitting tokens
            # Try consuming multiple tokens until the stripped concatenation matches
            concat = stripped_token
            scan_end = token_end
            matched = False
            
            # Try up to 3 extra tokens for cases like "p.m." → "p" + "." + "m" + "."
            for _ in range(3):
                if concat.lower() == fa_word:
                    spans.append((token_start, scan_end))
                    pos = scan_end
                    matched = True
                    break
                # Skip whitespace
                while scan_end < len(text) and text[scan_end].isspace():
                    scan_end += 1
                if scan_end >= len(text):
                    break
                # Consume next token
                next_start = scan_end
                while scan_end < len(text) and not text[scan_end].isspace():
                    scan_end += 1
                concat += _strip_punct(text[next_start:scan_end])
            
            if not matched:
                # Check final concatenation
                if concat.lower() == fa_word:
                    spans.append((token_start, scan_end))
                    pos = scan_end
                else:
                    # Fallback: skip this original token and try again
                    # (handles insertions/deletions between transcript and FA output)
                    spans.append((token_start, token_end))
                    pos = token_end
    
    return spans

## Word-to-VAD-Chunk Assignment

Each FA word has a `start_time`. We assign it to the VAD chunk whose time
range contains that timestamp. For words in silence gaps between chunks,
we assign to the nearest chunk by time proximity.

In [ ]:
#| export
def assign_words_to_chunks(
    fa_items: List[ForcedAlignItem],  # FA word-level alignment results
    vad_chunks: List[VADChunk],  # VAD chunks with start/end times
) -> List[int]:  # Chunk index for each FA word
    """Assign each FA word to a VAD chunk based on timestamp overlap.
    
    Words whose start_time falls within a chunk's [start, end] range are
    assigned to that chunk. Words in silence gaps are assigned to the
    nearest chunk by time proximity.
    """
    if not vad_chunks:
        return [0] * len(fa_items)
    
    assignments = []
    
    for item in fa_items:
        t = item.start_time
        best_idx = 0
        best_dist = float('inf')
        contained = False
        
        for i, chunk in enumerate(vad_chunks):
            # Strict containment (no tolerance)
            if chunk.start_time <= t <= chunk.end_time:
                best_idx = i
                contained = True
                break
            
            # Track nearest chunk for gap assignment
            dist = min(abs(t - chunk.start_time), abs(t - chunk.end_time))
            if dist < best_dist:
                best_dist = dist
                best_idx = i
        
        assignments.append(best_idx)
    
    return assignments

## Build TextSegments from Assignments

Groups words by their assigned chunk, joins the original (punctuated) text
spans, and creates `TextSegment` objects with proper character offsets.

In [ ]:
#| export
def build_segments_from_alignment(
    text: str,  # Original text with punctuation
    spans: List[Tuple[int, int]],  # Character spans from map_fa_words_to_text
    assignments: List[int],  # Chunk index per word from assign_words_to_chunks
    num_chunks: int,  # Total number of VAD chunks
    source_id: Optional[str] = None,  # Source block ID for traceability
    source_provider_id: Optional[str] = None,  # Source provider identifier
) -> List[TextSegment]:  # One segment per VAD chunk
    """Build TextSegment list by grouping words by their assigned VAD chunk.
    
    Each VAD chunk gets one TextSegment whose text is the joined original
    (punctuated) words assigned to that chunk.
    """
    # Group spans by chunk assignment
    chunk_spans: Dict[int, List[Tuple[int, int]]] = {}
    for span, chunk_idx in zip(spans, assignments):
        if chunk_idx not in chunk_spans:
            chunk_spans[chunk_idx] = []
        chunk_spans[chunk_idx].append(span)
    
    segments = []
    for chunk_idx in range(num_chunks):
        word_spans = chunk_spans.get(chunk_idx, [])
        
        if word_spans:
            # Extract text from original using spans
            # Use the full range from first word start to last word end
            seg_start = word_spans[0][0]
            seg_end = word_spans[-1][1]
            seg_text = text[seg_start:seg_end].strip()
        else:
            # Empty segment (VAD chunk with no assigned words)
            seg_text = ""
            seg_start = None
            seg_end = None
        
        segments.append(TextSegment(
            index=chunk_idx,
            text=seg_text,
            source_id=source_id,
            source_provider_id=source_provider_id,
            start_char=seg_start,
            end_char=seg_end,
        ))
    
    return segments

## ForcedAlignmentService

Wraps the forced alignment plugin to produce `List[TextSegment]` — the same
output type as `SegmentationService.split_sentences_async()`, so the downstream
UI flow is identical.

In [ ]:
#| export
class ForcedAlignmentService:
    """Service for audio-informed text pre-splitting via forced alignment plugin."""

    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager for accessing forced alignment plugin
        plugin_name: str = "cjm-transcription-plugin-qwen3-forced-aligner",  # Name of the FA plugin
    ):
        """Initialize the forced alignment service."""
        self._manager = plugin_manager
        self._plugin_name = plugin_name

    def is_available(self) -> bool:  # True if plugin is loaded and ready
        """Check if the forced alignment plugin is available."""
        return self._manager.get_plugin(self._plugin_name) is not None

    def ensure_loaded(
        self,
        config: Optional[Dict[str, Any]] = None,  # Optional plugin configuration
    ) -> bool:  # True if successfully loaded
        """Ensure the forced alignment plugin is loaded."""
        if self.is_available():
            return True
        meta = self._manager.get_discovered_meta(self._plugin_name)
        if meta:
            return self._manager.load_plugin(meta, config or {"language": "English"})
        return False

    async def align_and_split_async(
        self,
        audio_path: str,  # Path to the audio file
        text: str,  # Original transcript text blob (with punctuation)
        vad_chunks: List[VADChunk],  # VAD chunks for this audio
        source_id: Optional[str] = None,  # Source block ID for traceability
        source_provider_id: Optional[str] = None,  # Source provider identifier
    ) -> List[TextSegment]:  # One segment per VAD chunk
        """Run forced alignment and split text into segments matching VAD chunks."""
        if not self.is_available():
            raise RuntimeError(f"Plugin {self._plugin_name} not loaded")

        # 1. Call forced alignment plugin
        result = await self._manager.execute_plugin_async(
            self._plugin_name,
            audio=audio_path,
            text=text,
        )

        # Convert dict result to ForcedAlignItem objects
        items = [
            ForcedAlignItem(**item_dict)
            for item_dict in result.get("items", [])
        ]

        # 2. Map FA words back to original text positions
        spans = map_fa_words_to_text(text, items)

        # 3. Assign words to VAD chunks
        assignments = assign_words_to_chunks(items, vad_chunks)

        # 4-5. Group by chunk and build TextSegments
        segments = build_segments_from_alignment(
            text=text,
            spans=spans,
            assignments=assignments,
            num_chunks=len(vad_chunks),
            source_id=source_id,
            source_provider_id=source_provider_id,
        )

        return segments

    def align_and_split(
        self,
        audio_path: str,  # Path to the audio file
        text: str,  # Original transcript text blob
        vad_chunks: List[VADChunk],  # VAD chunks for this audio
        source_id: Optional[str] = None,
        source_provider_id: Optional[str] = None,
    ) -> List[TextSegment]:  # One segment per VAD chunk
        """Run forced alignment and split text synchronously."""
        return asyncio.get_event_loop().run_until_complete(
            self.align_and_split_async(
                audio_path, text, vad_chunks, source_id, source_provider_id
            )
        )

    async def align_and_split_combined_async(
        self,
        source_blocks: List[Any],  # SourceBlock objects with id, provider_id, text
        audio_paths: List[str],  # Audio file path per source block
        vad_chunks_by_source: List[List[VADChunk]],  # VAD chunks per source block
    ) -> List[TextSegment]:  # Combined segments with global indexing
        """Align and split multiple source blocks with their respective audio."""
        all_segments = []
        global_index = 0

        for block, audio_path, chunks in zip(source_blocks, audio_paths, vad_chunks_by_source):
            segments = await self.align_and_split_async(
                audio_path=audio_path,
                text=block.text,
                vad_chunks=chunks,
                source_id=block.id,
                source_provider_id=block.provider_id,
            )
            for seg in segments:
                seg.index = global_index
                global_index += 1
                all_segments.append(seg)

        return all_segments

## Tests

### Word-to-Text Mapping

In [ ]:
# Test _strip_punct
assert _strip_punct("hello") == "hello"
assert _strip_punct("hello,") == "hello"
assert _strip_punct("10th,") == "10th"
assert _strip_punct("p.m.") == "pm"
assert _strip_punct("I'm") == "Im"
assert _strip_punct('"Hello"') == "Hello"
print("_strip_punct tests passed")

_strip_punct tests passed


In [ ]:
# Test basic word-to-text mapping
text = "November the 10th, Wednesday, 9 p.m."
fa_items = [
    ForcedAlignItem(text="November", start_time=1.04, end_time=1.6),
    ForcedAlignItem(text="the", start_time=1.6, end_time=1.68),
    ForcedAlignItem(text="10th", start_time=1.76, end_time=2.08),
    ForcedAlignItem(text="Wednesday", start_time=2.48, end_time=3.04),
    ForcedAlignItem(text="9", start_time=3.84, end_time=4.16),
    ForcedAlignItem(text="pm", start_time=4.16, end_time=4.64),
]

spans = map_fa_words_to_text(text, fa_items)
print(f"Spans: {spans}")
for span, item in zip(spans, fa_items):
    print(f"  FA '{item.text}' → original '{text[span[0]:span[1]]}'")

# Verify punctuation preserved
assert text[spans[2][0]:spans[2][1]] == "10th,"  # trailing comma
assert text[spans[3][0]:spans[3][1]] == "Wednesday,"  # trailing comma
assert text[spans[5][0]:spans[5][1]] == "p.m."  # multi-char punctuation mapped to "pm"
print("Basic mapping tests passed")

Spans: [(0, 8), (9, 12), (13, 18), (19, 29), (30, 31), (32, 36)]
  FA 'November' → original 'November'
  FA 'the' → original 'the'
  FA '10th' → original '10th,'
  FA 'Wednesday' → original 'Wednesday,'
  FA '9' → original '9'
  FA 'pm' → original 'p.m.'
Basic mapping tests passed


In [ ]:
# Test with contractions
text2 = "I'm standing in a dark alley."
fa_items2 = [
    ForcedAlignItem(text="Im", start_time=0.0, end_time=0.2),
    ForcedAlignItem(text="standing", start_time=0.2, end_time=0.5),
    ForcedAlignItem(text="in", start_time=0.5, end_time=0.6),
    ForcedAlignItem(text="a", start_time=0.6, end_time=0.7),
    ForcedAlignItem(text="dark", start_time=0.7, end_time=0.9),
    ForcedAlignItem(text="alley", start_time=0.9, end_time=1.2),
]

spans2 = map_fa_words_to_text(text2, fa_items2)
for span, item in zip(spans2, fa_items2):
    print(f"  FA '{item.text}' → original '{text2[span[0]:span[1]]}'")

assert text2[spans2[0][0]:spans2[0][1]] == "I'm"  # contraction preserved
assert text2[spans2[5][0]:spans2[5][1]] == "alley."  # trailing period
print("Contraction mapping tests passed")

  FA 'Im' → original 'I'm'
  FA 'standing' → original 'standing'
  FA 'in' → original 'in'
  FA 'a' → original 'a'
  FA 'dark' → original 'dark'
  FA 'alley' → original 'alley.'
Contraction mapping tests passed


### Word-to-VAD-Chunk Assignment

In [ ]:
# Test basic chunk assignment (using realistic VAD data with gaps)
chunks = [
    VADChunk(index=0, start_time=1.1, end_time=2.4),
    VADChunk(index=1, start_time=2.5, end_time=3.2),
    VADChunk(index=2, start_time=3.8, end_time=4.7),
]

# Words with timestamps from forced alignment
items = [
    ForcedAlignItem(text="November", start_time=1.04, end_time=1.6),   # gap before chunk 0 → nearest chunk 0
    ForcedAlignItem(text="the", start_time=1.6, end_time=1.68),        # chunk 0
    ForcedAlignItem(text="10th", start_time=1.76, end_time=2.08),      # chunk 0
    ForcedAlignItem(text="Wednesday", start_time=2.48, end_time=3.04), # gap [2.4,2.5] → nearest chunk 1
    ForcedAlignItem(text="9", start_time=3.84, end_time=4.16),        # chunk 2
    ForcedAlignItem(text="pm", start_time=4.16, end_time=4.64),       # chunk 2
]

assignments = assign_words_to_chunks(items, chunks)
print(f"Assignments: {assignments}")
assert assignments == [0, 0, 0, 1, 2, 2]
print("Basic chunk assignment tests passed")

Assignments: [0, 0, 0, 1, 2, 2]
Basic chunk assignment tests passed


In [ ]:
# Test word in silence gap (between chunks)
gap_chunks = [
    VADChunk(index=0, start_time=1.0, end_time=2.0),
    VADChunk(index=1, start_time=4.0, end_time=5.0),
]

gap_items = [
    ForcedAlignItem(text="word1", start_time=1.5, end_time=1.8),   # chunk 0
    ForcedAlignItem(text="word2", start_time=2.5, end_time=3.0),   # gap → nearest is chunk 0 (dist 0.5 vs 1.0)
    ForcedAlignItem(text="word3", start_time=3.5, end_time=3.8),   # gap → nearest is chunk 1 (dist 0.5 vs 1.5)
    ForcedAlignItem(text="word4", start_time=4.5, end_time=4.8),   # chunk 1
]

gap_assignments = assign_words_to_chunks(gap_items, gap_chunks)
print(f"Gap assignments: {gap_assignments}")
assert gap_assignments == [0, 0, 1, 1]
print("Gap assignment tests passed")

Gap assignments: [0, 0, 1, 1]
Gap assignment tests passed


### Build Segments from Alignment

In [ ]:
# Test full pipeline: map → assign → build
full_text = "November the 10th, Wednesday, 9 p.m."
full_items = [
    ForcedAlignItem(text="November", start_time=1.04, end_time=1.6),
    ForcedAlignItem(text="the", start_time=1.6, end_time=1.68),
    ForcedAlignItem(text="10th", start_time=1.76, end_time=2.08),
    ForcedAlignItem(text="Wednesday", start_time=2.48, end_time=3.04),
    ForcedAlignItem(text="9", start_time=3.84, end_time=4.16),
    ForcedAlignItem(text="pm", start_time=4.16, end_time=4.64),
]
full_chunks = [
    VADChunk(index=0, start_time=1.1, end_time=2.4),
    VADChunk(index=1, start_time=2.5, end_time=3.2),
    VADChunk(index=2, start_time=3.8, end_time=4.7),
]

full_spans = map_fa_words_to_text(full_text, full_items)
full_assignments = assign_words_to_chunks(full_items, full_chunks)
segments = build_segments_from_alignment(
    full_text, full_spans, full_assignments, len(full_chunks),
    source_id="test-source", source_provider_id="test-provider"
)

print(f"Built {len(segments)} segments:")
for seg in segments:
    print(f"  [{seg.index}] '{seg.text}' (chars {seg.start_char}-{seg.end_char})")

assert len(segments) == 3
assert segments[0].text == "November the 10th,"
assert segments[1].text == "Wednesday,"
assert segments[2].text == "9 p.m."
assert segments[0].source_id == "test-source"
assert segments[0].source_provider_id == "test-provider"
assert segments[0].index == 0
assert segments[1].index == 1
assert segments[2].index == 2
print("Build segments tests passed")

Built 3 segments:
  [0] 'November the 10th,' (chars 0-18)
  [1] 'Wednesday,' (chars 19-29)
  [2] '9 p.m.' (chars 30-36)
Build segments tests passed


In [ ]:
# Test with a VAD chunk that has no assigned words
sparse_chunks = [
    VADChunk(index=0, start_time=1.0, end_time=2.0),
    VADChunk(index=1, start_time=5.0, end_time=6.0),  # No words fall here
    VADChunk(index=2, start_time=10.0, end_time=11.0),
]
sparse_items = [
    ForcedAlignItem(text="hello", start_time=1.5, end_time=1.8),
    ForcedAlignItem(text="world", start_time=10.5, end_time=10.8),
]
sparse_text = "hello world"

sparse_spans = map_fa_words_to_text(sparse_text, sparse_items)
sparse_assign = assign_words_to_chunks(sparse_items, sparse_chunks)
sparse_segments = build_segments_from_alignment(
    sparse_text, sparse_spans, sparse_assign, len(sparse_chunks)
)

assert len(sparse_segments) == 3
assert sparse_segments[0].text == "hello"
assert sparse_segments[1].text == ""  # Empty segment for chunk with no words
assert sparse_segments[2].text == "world"
print(f"Sparse segments: {[(s.index, s.text) for s in sparse_segments]}")
print("Empty chunk segment tests passed")

Sparse segments: [(0, 'hello'), (1, ''), (2, 'world')]
Empty chunk segment tests passed


### ForcedAlignmentService with Qwen3 Plugin

These tests require the Qwen3 forced alignment plugin to be installed and discoverable.
VAD chunks are loaded from pre-generated JSON files in `test_files/`.

In [ ]:
#| eval: false
# Set up plugin manager and load the Qwen3 forced alignment plugin
import json
from pathlib import Path
from cjm_plugin_system.core.manager import PluginManager

# Calculate project root from notebook location (nbs/services/ -> project root)
project_root = Path.cwd().parent.parent
test_files = project_root / "test_files"
manifests_dir = project_root / ".cjm" / "manifests"

# Create plugin manager with explicit search path
manager = PluginManager(search_paths=[manifests_dir])
manager.discover_manifests()

print(f"Discovered {len(manager.discovered)} plugins from {manifests_dir}")

# Check if Qwen3 forced alignment plugin is available
fa_plugin_name = "cjm-transcription-plugin-qwen3-forced-aligner"
fa_meta = manager.get_discovered_meta(fa_plugin_name)
if fa_meta:
    print(f"Found plugin: {fa_meta.name} v{fa_meta.version}")
    manager.load_plugin(fa_meta, {"language": "English"})
    print(f"  {fa_plugin_name}: loaded")
else:
    print(f"  {fa_plugin_name}: not found - install via plugins_test.yaml")

[PluginManager] Discovered manifest: cjm-media-plugin-silero-vad from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcript-segment-align/.cjm/manifests/cjm-media-plugin-silero-vad.json
[PluginManager] Discovered manifest: cjm-text-plugin-nltk from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcript-segment-align/.cjm/manifests/cjm-text-plugin-nltk.json
[PluginManager] Discovered manifest: cjm-transcription-plugin-qwen3-forced-aligner from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcript-segment-align/.cjm/manifests/cjm-transcription-plugin-qwen3-forced-aligner.json
[PluginManager] Launching worker for cjm-transcription-plugin-qwen3-forced-aligner...


Discovered 3 plugins from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcript-segment-align/.cjm/manifests
Found plugin: cjm-transcription-plugin-qwen3-forced-aligner v0.0.1
[cjm-transcription-plugin-qwen3-forced-aligner] Starting worker on port 41997...
[cjm-transcription-plugin-qwen3-forced-aligner] Logs: /home/innom-dt/.cjm/logs/cjm-transcription-plugin-qwen3-forced-aligner.log


[PluginManager] HTTP Request: GET http://127.0.0.1:41997/health "HTTP/1.1 200 OK"
[PluginManager] HTTP Request: POST http://127.0.0.1:41997/initialize "HTTP/1.1 200 OK"
[PluginManager] Loaded plugin: cjm-transcription-plugin-qwen3-forced-aligner


[cjm-transcription-plugin-qwen3-forced-aligner] Worker ready.
  cjm-transcription-plugin-qwen3-forced-aligner: loaded


In [ ]:
#| eval: false
# Test with short audio file
if fa_meta and manager.get_plugin(fa_plugin_name):
    # Load VAD chunks from pre-generated JSON
    with open(test_files / "short_test_audio_vad.json") as f:
        vad_data = json.load(f)
    vad_chunks = [VADChunk(**c) for c in vad_data]

    # Load transcript text
    text = (test_files / "short_test_audio.txt").read_text().strip()
    audio_path = str(test_files / "short_test_audio.mp3")

    print(f"Audio: short_test_audio.mp3")
    print(f"Text: {text[:60]}...")
    print(f"VAD chunks: {len(vad_chunks)}")

    # Create service and run alignment
    fa_service = ForcedAlignmentService(manager, fa_plugin_name)

    segments = await fa_service.align_and_split_async(
        audio_path=audio_path,
        text=text,
        vad_chunks=vad_chunks,
        source_id="test-short",
        source_provider_id="test",
    )

    print(f"\nResult: {len(segments)} segments (from {len(vad_chunks)} VAD chunks)")
    assert len(segments) == len(vad_chunks), f"Expected {len(vad_chunks)} segments, got {len(segments)}"

    for seg in segments:
        print(f"  [{seg.index}] '{seg.text[:50]}{'...' if len(seg.text) > 50 else ''}'")

    # Verify all text is accounted for (no words lost)
    all_text = " ".join(seg.text for seg in segments if seg.text)
    print(f"\nReconstructed text length: {len(all_text)} chars")
    print(f"Original text length: {len(text)} chars")

Audio: short_test_audio.mp3
Text: November the 10th, Wednesday, 9 p.m. I'm standing in a dark ...
VAD chunks: 11


[PluginManager] HTTP Request: POST http://127.0.0.1:41997/execute "HTTP/1.1 200 OK"



Result: 11 segments (from 11 VAD chunks)
  [0] 'November the 10th,'
  [1] 'Wednesday,'
  [2] '9 p.m.'
  [3] 'I'm standing in a dark alley.'
  [4] 'After waiting several hours,'
  [5] 'the time has come.'
  [6] 'A woman with long dark hair approaches.'
  [7] 'I have to act'
  [8] 'and fast'
  [9] 'before she realises what has happened.'
  [10] 'I must find out.'

Reconstructed text length: 233 chars
Original text length: 233 chars


In [ ]:
#| eval: false
# Test with longer audio file (Laying Plans ~4 min, 89 VAD chunks)
if fa_meta and manager.get_plugin(fa_plugin_name):
    with open(test_files / "02 - 1. Laying Plans_vad.json") as f:
        vad_data_long = json.load(f)
    vad_chunks_long = [VADChunk(**c) for c in vad_data_long]

    text_long = (test_files / "02 - 1. Laying Plans.txt").read_text().strip()
    audio_path_long = str(test_files / "02 - 1. Laying Plans.mp3")

    print(f"Audio: 02 - 1. Laying Plans.mp3")
    print(f"Text: {len(text_long)} chars, {len(text_long.split())} words")
    print(f"VAD chunks: {len(vad_chunks_long)}")

    segments_long = await fa_service.align_and_split_async(
        audio_path=audio_path_long,
        text=text_long,
        vad_chunks=vad_chunks_long,
        source_id="test-long",
        source_provider_id="test",
    )

    print(f"\nResult: {len(segments_long)} segments (from {len(vad_chunks_long)} VAD chunks)")
    assert len(segments_long) == len(vad_chunks_long)

    # Show first and last few segments
    print("\nFirst 5 segments:")
    for seg in segments_long[:5]:
        print(f"  [{seg.index}] '{seg.text[:60]}{'...' if len(seg.text) > 60 else ''}'")
    print(f"\nLast 5 segments:")
    for seg in segments_long[-5:]:
        print(f"  [{seg.index}] '{seg.text[:60]}{'...' if len(seg.text) > 60 else ''}'")

    # Check for empty segments
    empty_count = sum(1 for s in segments_long if not s.text)
    print(f"\nEmpty segments: {empty_count}")
    non_empty_text = " ".join(s.text for s in segments_long if s.text)
    print(f"Reconstructed text: {len(non_empty_text)} chars (original: {len(text_long)} chars)")

Audio: 02 - 1. Laying Plans.mp3
Text: 3405 chars, 599 words
VAD chunks: 89


[PluginManager] HTTP Request: POST http://127.0.0.1:41997/execute "HTTP/1.1 200 OK"



Result: 89 segments (from 89 VAD chunks)

First 5 segments:
  [0] 'Laying Plans'
  [1] 'Sun Tzu said,'
  [2] 'The art of war is of vital importance to the state.'
  [3] 'It is a matter of life and death,'
  [4] 'a road either to safety or to ruin.'

Last 5 segments:
  [84] 'Thus do many calculations lead to victory'
  [85] 'and few calculations to defeat.'
  [86] 'How much more no calculation at all? It'
  [87] 'is by attention to this point that I can foresee who is like...'
  [88] 'or lose.'

Empty segments: 0
Reconstructed text: 3405 chars (original: 3405 chars)


In [ ]:
#| eval: false
# Cleanup
if fa_meta:
    manager.unload_all()
    print("Plugins unloaded")

[PluginManager] HTTP Request: POST http://127.0.0.1:41997/cleanup "HTTP/1.1 200 OK"
[PluginManager] Unloaded plugin: cjm-transcription-plugin-qwen3-forced-aligner


Plugins unloaded


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()